# 第 2 课：工具调用 + 手写 Agent 循环

第 1 课是一问一答，这一课让模型**自己决定去查什么**。场景就是架构图里那个例子：“订单 1024 为什么还没发货？”

**这一课学到的**
- LLM：工具定义（JSON Schema）、tool_calls、手写 Agent 循环、步数上限、工具报错的处理
- Python：函数和 docstring、`**kwargs` 解包、字典当“方法表”、`json`

**核心事实：模型从不直接碰你的数据。** 它只“说”要调哪个工具、传什么参数，真正执行的是你的代码。

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()  # 读取同目录的 .env 文件，相当于 appsettings.json + 环境变量

client = OpenAI(
    api_key=os.environ["LLM_API_KEY"],
    base_url=os.getenv("LLM_BASE_URL", "https://open.bigmodel.cn/api/paas/v4/"),
)
MODEL = os.getenv("LLM_MODEL", "glm-4-flash")
print("使用模型:", MODEL)

## 1. 假数据和工具函数

工具就是普通 Python 函数。这里用 list + dict 模拟数据库，真实项目里换成查数据库或调接口即可。

In [ ]:
ORDERS = [
    {"order_id": "1024", "customer": "上海星辰软件有限公司", "amount": 5000, "status": "待发货",
     "payment_status": "待确认：银行转账已提交，等待财务核对到账"},
    {"order_id": "1025", "customer": "上海星辰软件有限公司", "amount": 1280, "status": "已发货", "payment_status": "已支付"},
    {"order_id": "1026", "customer": "杭州云帆科技有限公司", "amount": 860, "status": "已完成", "payment_status": "已支付"},
]
SHIPMENTS = {
    "1025": {"carrier": "顺丰", "tracking_no": "SF1234567890", "location": "杭州转运中心，预计明天送达"},
    "1026": {"carrier": "中通", "tracking_no": "ZT9876543210", "location": "已签收"},
}


def get_order(order_id: str) -> dict | str:
    """按订单号查询订单。"""
    # next(..., 默认值) 相当于 C# 的 FirstOrDefault()
    return next((o for o in ORDERS if o["order_id"] == order_id), f"找不到订单 {order_id}")


def get_shipment(order_id: str) -> dict | str:
    """按订单号查询物流。"""
    return SHIPMENTS.get(order_id, f"订单 {order_id} 还没有物流信息（未发货）")


def find_orders_by_customer(customer_name: str) -> list[str]:
    """按客户名称（可以只写一部分）查订单号。"""
    # 列表推导式，相当于 C# 的 .Where(...).Select(...).ToList()
    return [o["order_id"] for o in ORDERS if customer_name in o["customer"]]


print(get_order("1024"))
print(find_orders_by_customer("星辰"))

## 2. 告诉模型有哪些工具

模型看不到你的代码，只能看到下面这份**工具说明书**（JSON Schema）。`description` 写得好不好，直接决定模型会不会用、用得对不对，就像写对外 API 文档。

In [ ]:
def tool(name: str, description: str, params: dict[str, str]) -> dict:
    """生成一个工具定义。params 是 {参数名: 参数说明}，这里都按字符串参数处理。"""
    return {
        "type": "function",
        "function": {
            "name": name,
            "description": description,
            "parameters": {
                "type": "object",
                "properties": {p: {"type": "string", "description": d} for p, d in params.items()},
                "required": list(params),
            },
        },
    }


TOOLS = [
    tool("get_order", "按订单号查询订单详情：客户、金额、订单状态、支付状态", {"order_id": "订单号，例如 1024"}),
    tool("get_shipment", "按订单号查询物流信息。订单还没发货时会说明未发货", {"order_id": "订单号，例如 1024"}),
    tool("find_orders_by_customer", "按客户名称（可以只写一部分）查询该客户的所有订单号", {"customer_name": "客户名称，例如 星辰"}),
]

# 工具名 → 真正执行的函数。字典里存函数，相当于 C# 的 Dictionary<string, Func<...>>
TOOL_IMPL = {
    "get_order": get_order,
    "get_shipment": get_shipment,
    "find_orders_by_customer": find_orders_by_customer,
}

import json
print(json.dumps(TOOLS[0], ensure_ascii=False, indent=2))

## 3. 核心：手写 Agent 循环

整个 Agent 的骨架就是这 5 步，循环到模型不再要求调工具为止：

1. 把对话历史 + 工具清单发给模型
2. 看回复里有没有 `tool_calls`
3. 没有 → 模型答完了，结束
4. 有 → **我们的代码**执行工具（权限检查、日志、人工审批都加在这里）
5. 把工具结果作为 `role: "tool"` 的消息放回对话，回到第 1 步

In [ ]:
import json

SYSTEM = "你是订单客服助手。回答前必须用工具查询真实数据，不要编造。回答简洁，说明原因和下一步建议。"


def run_agent(question: str, max_steps: int = 5) -> tuple[str, list[dict]]:
    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": question},
    ]

    for step in range(1, max_steps + 1):
        # ① 发给模型
        resp = client.chat.completions.create(model=MODEL, messages=messages, tools=TOOLS, temperature=0)
        msg = resp.choices[0].message

        # 模型这一轮的回复也要记进历史（包括它要调用哪些工具）
        assistant = {"role": "assistant", "content": msg.content or ""}
        if msg.tool_calls:
            assistant["tool_calls"] = [
                {"id": c.id, "type": "function", "function": {"name": c.function.name, "arguments": c.function.arguments}}
                for c in msg.tool_calls
            ]
        messages.append(assistant)

        # ②③ 没有工具调用 = 答完了
        if not msg.tool_calls:
            return msg.content, messages

        # ④ 我们来执行工具
        for call in msg.tool_calls:
            name = call.function.name
            args = json.loads(call.function.arguments or "{}")   # 模型给的参数是 JSON 字符串
            print(f"  🔧 第 {step} 步：调用 {name}({args})")

            func = TOOL_IMPL.get(name)
            if func is None:
                result = f"错误：没有名为 {name} 的工具"
            else:
                try:
                    result = func(**args)   # **args 把 dict 展开成命名参数：func(order_id="1024")
                except Exception as e:      # 工具出错不要让程序崩，把错误告诉模型
                    result = f"工具执行出错：{e}"
            print(f"     ↳ {result}")

            # ⑤ 结果放回对话，tool_call_id 用来对应是哪一次调用
            messages.append({
                "role": "tool",
                "tool_call_id": call.id,
                "content": json.dumps(result, ensure_ascii=False),
            })

    return "（达到最大步数，已停止）", messages

## 4. 试一试

In [ ]:
answer, history = run_agent("订单 1024 为什么还没发货？")
print("\n🤖", answer)

In [ ]:
answer, history = run_agent("星辰软件一共有几个订单，各是什么状态？")
print("\n🤖", answer)

## 5. 看看完整的对话历史

这就是模型每一轮实际看到的全部内容。Agent Framework 里的 `Thread` / `Session`，本质上就是在帮你管理这个列表。

In [ ]:
for m in history:
    role = m["role"]
    if m.get("tool_calls"):
        print(f"[{role}] 请求调用：", [c["function"]["name"] + c["function"]["arguments"] for c in m["tool_calls"]])
    else:
        print(f"[{role}]", m["content"][:120])

## 动手练习

1. 把 `get_order` 的 `description` 改成含糊的“查数据”，看模型还会不会正确调用。
2. 加一个**写操作**工具 `cancel_order(order_id, reason)`，在第 ④ 步执行它之前用 `input("确认取消？(y/n)")` 让你确认，这就是 human-in-the-loop。
3. 故意让 `get_shipment` 抛异常（`raise RuntimeError("物流系统超时")`），看模型怎么处理工具报错。
4. 把 `max_steps` 改成 1，问一个需要连续查两次的问题，看会发生什么。